# 🏭 Project 3.4 — Predictive Maintenance
**DSA for Mechatronics · Week 03**

> **Your task:** Run every cell from top to bottom.  
> At the end, a full report is printed automatically.  
> **Submit:** A screenshot or PDF of the final report cell output.  
> **Present:** Be ready to explain what each step does and why.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ║   This seeds your personal dataset — be honest!     ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

# ── seed everything from your ID ──────────────────────
import random, hashlib
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready — run the next cells.")


## Step 1 — Generate your personal vibration dataset

In [ ]:
# CNC spindle vibration — 10 minutes at 1 Hz
N              = 600
BEARING_PERIOD = random.randint(12, 40)      # hidden periodic fault
THRESHOLD      = round(random.uniform(4.5, 6.5), 1)   # fault alarm level
MIN_DURATION   = random.randint(4, 8)        # min seconds to count as a fault event

# Two fault windows at random positions
fw1_start = random.randint(60,  200)
fw1_end   = fw1_start + random.randint(10, 25)
fw2_start = random.randint(fw1_end + 60, 450)
fw2_end   = fw2_start + random.randint(8, 20)

vibration = []
for t in range(N):
    base    = max(0.1, 1.0 + random.gauss(0, 0.35))
    bearing = random.uniform(THRESHOLD + 0.5, THRESHOLD + 4.0) if t % BEARING_PERIOD == 0 else 0
    fault   = 0
    if fw1_start <= t <= fw1_end: fault = random.uniform(THRESHOLD + 1.0, THRESHOLD + 5.0)
    if fw2_start <= t <= fw2_end: fault = random.uniform(THRESHOLD + 1.5, THRESHOLD + 5.5)
    vibration.append(round(base + bearing + fault, 3))

print("Vibration signal parameters:")
print(f"  Samples        : {N}  (10 minutes at 1 Hz)")
print(f"  Alarm threshold: {THRESHOLD} mm/s")
print(f"  Min fault dur. : {MIN_DURATION} s")
print(f"  Bearing period : ??? (you will calculate this in Step 5)")
print(f"  Fault windows  : t={fw1_start}–{fw1_end}s  and  t={fw2_start}–{fw2_end}s")
print()
print("First 20 readings (mm/s):")
for t, v in enumerate(vibration[:20]):
    bar = '█' * int(v * 3)
    spike = " ← BEARING SPIKE" if t % BEARING_PERIOD == 0 else ""
    print(f"  t={t:03d}: {v:6.3f}  {bar}{spike}")


## Step 2 — Build prefix sum for O(1) range queries

In [ ]:
def build_prefix_sum(signal):
    """
    prefix[0]   = 0
    prefix[i+1] = prefix[i] + signal[i]
    Range sum signal[a..b] = prefix[b+1] - prefix[a]  →  O(1)
    """
    prefix = [0.0]
    for v in signal:
        prefix.append(prefix[-1] + v)
    return prefix

def range_avg(prefix, a, b):
    """Average of signal[a..b] inclusive in O(1)."""
    return (prefix[b + 1] - prefix[a]) / (b - a + 1)

prefix = build_prefix_sum(vibration)

print("Prefix sum built ✅")
print(f"  Prefix array length: {len(prefix)}  (signal length + 1)")
print()
print("Sample range queries (O(1) each):")
queries = [(0, 59), (fw1_start, fw1_end), (fw2_start, fw2_end), (0, N-1)]
labels  = ["First 60s (normal)", f"Fault window 1", f"Fault window 2", "Full recording"]
for (a, b), label in zip(queries, labels):
    avg = range_avg(prefix, a, b)
    print(f"  {label:<22}: t={a:03d}–{b:03d}  avg={avg:.3f} mm/s")


## Step 3 — Detect sustained fault windows (sliding window scan)

In [ ]:
def find_fault_windows(signal, threshold, min_duration):
    """
    Detect contiguous segments where signal stays above threshold
    for at least min_duration consecutive seconds.
    Returns list of event dicts.
    """
    events = []
    i = 0
    while i < len(signal):
        if signal[i] > threshold:
            j = i
            while j < len(signal) and signal[j] > threshold:
                j += 1
            duration = j - i
            if duration >= min_duration:
                window = signal[i:j]
                events.append({
                    'start':    i,
                    'end':      j - 1,
                    'duration': duration,
                    'peak':     round(max(window), 3),
                    'avg':      round(sum(window) / duration, 3),
                })
            i = j
        else:
            i += 1
    return events

events = find_fault_windows(vibration, THRESHOLD, MIN_DURATION)

def fmt(t): return f"{t//60:02d}:{t%60:02d}"

print(f"Fault window detection (threshold={THRESHOLD} mm/s, min={MIN_DURATION}s):")
print(f"  Events found: {len(events)}")
print()
for k, e in enumerate(events, 1):
    print(f"  Event {k}:")
    print(f"    Time     : {fmt(e['start'])} – {fmt(e['end'])}  ({e['duration']} s)")
    print(f"    Peak     : {e['peak']} mm/s")
    print(f"    Average  : {e['avg']} mm/s")


## Step 4 — Estimate the bearing fault period (frequency counting)

In [ ]:
def detect_bearing_period(signal, spike_threshold, min_gap=5):
    """
    1. Find all spike indices above spike_threshold
    2. Compute gaps between consecutive spikes
    3. Filter gaps < min_gap (noise / fault-window artefacts)
    4. Return the most common gap — the bearing period
    """
    spikes = [t for t, v in enumerate(signal) if v > spike_threshold]
    gaps   = [spikes[i+1] - spikes[i] for i in range(len(spikes) - 1)]
    gaps   = [g for g in gaps if g >= min_gap]
    freq   = {}
    for g in gaps:
        freq[g] = freq.get(g, 0) + 1
    return max(freq, key=freq.get), freq

est_period, gap_freq = detect_bearing_period(vibration, THRESHOLD)
top_gaps = sorted(gap_freq.items(), key=lambda x: -x[1])[:6]

print(f"Bearing period estimation:")
print(f"  Spike threshold used : {THRESHOLD} mm/s")
print(f"  Total spikes found   : {sum(1 for v in vibration if v > THRESHOLD)}")
print()
print(f"  Most common inter-spike gaps:")
for gap, count in top_gaps:
    bar = '█' * count
    print(f"    {gap:>4} s  ×{count:<3}  {bar}")
print()
print(f"  Estimated bearing period : {est_period} s")
print(f"  Actual bearing period    : {BEARING_PERIOD} s  ← revealed!")
print(f"  Error                    : {abs(est_period - BEARING_PERIOD)} s")


## 📋 Final Report — this is what you submit

In [ ]:
def fmt(t): return f"{t//60:02d}:{t%60:02d}"

total_fault_time = sum(e['duration'] for e in events)
overall_avg      = range_avg(prefix, 0, N - 1)
n_ev             = len(events)
status           = "🟢 NORMAL" if n_ev == 0 else ("🟡 CAUTION" if n_ev == 1 else "🔴 FAULT")

W = 58
print("╔" + "═"*W + "╗")
print("║" + " CNC SPINDLE — PREDICTIVE MAINTENANCE REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<26}: {STUDENT_ID:<28}║")
print(f"║  {'Dataset seed':<26}: {_seed:<28}║")
print("╠" + "═"*W + "╣")
print("║  MONITORING SUMMARY" + " "*(W-20) + "║")
print(f"║  {'Duration':<26}: 10 min  (600 readings){'':<14}║")
print(f"║  {'Overall avg vibration':<26}: {overall_avg:.3f} mm/s{'':<22}║")
print(f"║  {'Alarm threshold':<26}: {THRESHOLD} mm/s{'':<26}║")
print(f"║  {'Min fault duration':<26}: {MIN_DURATION} s{'':<29}║")
print("╠" + "═"*W + "╣")
print("║  FAULT EVENTS" + " "*(W-14) + "║")
if events:
    for k, e in enumerate(events, 1):
        print(f"║  Event {k}: {fmt(e['start'])}–{fmt(e['end'])}  "
              f"({e['duration']}s)  peak={e['peak']} mm/s  avg={e['avg']} mm/s{'':>{W-56}}║")
else:
    print(f"║  {'No fault events detected':<{W}}║")
print(f"║  {'Total fault time':<26}: {total_fault_time} s{'':<29}║")
print("╠" + "═"*W + "╣")
print("║  BEARING HEALTH" + " "*(W-16) + "║")
print(f"║  {'Estimated period':<26}: {est_period} s{'':<30}║")
print(f"║  {'Actual period':<26}: {BEARING_PERIOD} s{'':<30}║")
print(f"║  {'Estimation error':<26}: {abs(est_period-BEARING_PERIOD)} s{'':<30}║")
print("╠" + "═"*W + "╣")
print(f"║  STATUS : {status:<{W-11}}║")
print("╚" + "═"*W + "╝")
